<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/python/notebooks/c2_l9.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C2-L9 · Proyecto: pipeline end-to-end
ETH 90 días: carga → señal SMA(10,30) → costos → métricas → walk-forward → veredicto en una página.

In [ ]:
# CELDA COLAB-FIRST: correla primero si estas en Google Colab.
# Descarga el CSV del repo; si falla (sin red), usa el CSV local.
import pandas as pd
from pathlib import Path

ORG = "Emelecto"  # organizacion fija del repo Emelecto/QuantLab
CSV_NOMBRE = "c2_l9_eth.csv"
CSV_URL = f"https://raw.githubusercontent.com/{ORG}/QuantLab/main/web/content/cursos/python/data/{CSV_NOMBRE}"

try:
    df = pd.read_csv(CSV_URL)
    print("CSV descargado desde:", CSV_URL)
except Exception as e:
    print("Uso CSV local (motivo:", str(e)[:80] + ")")
    csv_path = Path("../data") / CSV_NOMBRE
    if not csv_path.exists():
        csv_path = Path(CSV_NOMBRE)  # fallback si corres desde data/
    df = pd.read_csv(csv_path)
print(df.shape)
print(df.head())

In [ ]:
FAST, SLOW = 10, 30
import numpy as np

COSTO = 0.0005  # 5 bps por cambio de posicion
df["ret"] = np.log(df["precio"] / df["precio"].shift(1)).fillna(0)
df["sma_fast"] = df["precio"].rolling(FAST).mean()
df["sma_slow"] = df["precio"].rolling(SLOW).mean()
df["pos"] = (df["sma_fast"].shift(1) > df["sma_slow"].shift(1)).astype(int).fillna(0)
df["bruto"] = df["pos"].shift(1).fillna(0) * df["ret"]
df["costo"] = df["pos"].diff().abs().fillna(0) * COSTO
df["neto"] = df["bruto"] - df["costo"]
df["equity"] = np.exp(df["neto"].cumsum())
trades = int(df["pos"].diff().abs().sum())
print(f"trades: {trades}  costo total: {df['costo'].sum():.4f} ({df['costo'].sum()*10000:.0f} bps)")
print(f"ret bruto: {np.exp(df['bruto'].sum())-1:+.2%}  ret neto: {df['equity'].iloc[-1]-1:+.2%}")
r = df["neto"]
sharpe = r.mean() / r.std(ddof=0) * (252 ** 0.5)
pico = df["equity"].cummax()
dd = df["equity"] / pico - 1
print(f"Sharpe neto anualizado: {sharpe:.2f}")
print(f"MaxDD: {dd.min():.2%}")

## El pipeline completo
Tres ventanas walk-forward (entrena 45, prueba 15) con grid 3×3. Si el OOS no aguanta, el veredicto es no operar.

In [ ]:
GRID = [(5, 20), (5, 30), (5, 40), (10, 20), (10, 30), (10, 40), (15, 20), (15, 30), (15, 40)]
import math
res = []
px = df["precio"].tolist()
def bt_net(p, f, l):
    n = len(p)
    r = [math.log(b/a) for a, b in zip(p[:-1], p[1:])]
    pos = [0]*n
    for t in range(l, n):
        pos[t] = 1 if sum(p[t-f:t])/f > sum(p[t-l:t])/l else 0
    net = [0.0]*n
    for t in range(1, n):
        net[t] = pos[t-1]*r[t-1] - 0.0005*abs(pos[t]-pos[t-1])
    return net
def bts(net):
    m = sum(net)/len(net)
    sd = (sum((x-m)**2 for x in net)/len(net)) ** 0.5
    return m/sd*(252**0.5) if sd > 0 else 0.0
for s in [0, 15, 30]:
    tr, ext = px[s:s+45], px[s:s+60]  # el test usa contexto trailing
    scored = sorted(((bts(bt_net(tr, f, l)), f, l) for f, l in GRID), reverse=True)
    bi, bf, bl = scored[0]
    bo = bts(bt_net(ext, bf, bl)[45:])
    res.append((bf, bl, round(bi, 3), round(bo, 3), round(bi-bo, 3)))
    print(f"ventana s={s}: mejor=({bf},{bl}) IS={bi:+.3f} OOS={bo:+.3f} gap={bi-bo:+.3f}")


In [ ]:
import matplotlib.pyplot as plt

fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
a1.plot(df["precio"], color="#5eead4")
a1.set_ylabel("ETH")
a1.set_title("Pipeline end-to-end: precio y equity neto")
a2.plot(df["equity"], color="#f59e0b")
a2.set_ylabel("equity")
a2.set_xlabel("día")
fig.tight_layout()
plt.show()

In [ ]:
# Chequeos automáticos
assert len(df) == 90, "se esperan 90 días"
assert trades == 4, "nº de trades determinista"
assert len(res) == 3, "3 ventanas walk-forward"
print("OK: pipeline end-to-end verificado")